# 02.03 — Agregaciones, Joins y Window Functions

## Objetivo

En este notebook trabajaremos con operaciones que permiten resumir, relacionar y analizar información entre diferentes grupos de datos.

Los principales temas serán:

- agregaciones con `groupBy()`;
- múltiples funciones de agregación;
- agregaciones condicionales;
- joins entre DataFrames;
- tipos de join;
- manejo de columnas ambiguas;
- Window Functions;
- rankings;
- acumulados;
- `lag()` y `lead()`.

Estas operaciones son fundamentales en procesos ETL, análisis de datos y construcción de variables para modelos.


## 1. Preparar el entorno

Comenzamos importando las herramientas necesarias y creando nuestra `SparkSession`.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType,
    DateType
)

spark = (
    SparkSession.builder
    .appName("02_03_aggregations_joins_windows")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")

Spark version: 4.1.3
Master: local[*]


## 2. Datasets de trabajo

Trabajaremos con dos conjuntos de datos relacionados:

```text
clientes
    │
    │ id_cliente
    │
    └──────────────┐
                   │
                   ▼
             transacciones
```

Un cliente puede tener múltiples transacciones, por lo que tendremos una relación:

```text
Cliente 1 ──► Transacción A
          ├─► Transacción B
          └─► Transacción C
```

Esta estructura nos permitirá practicar tanto agregaciones como joins y funciones de ventana.

### 2.1 DataFrame de clientes

In [2]:
clientes_data = [
    (1, "Ana",    "CDMX",        "Premium"),
    (2, "Luis",   "Jalisco",     "Básico"),
    (3, "Marta",  "Nuevo León",  "Premium"),
    (4, "Carlos", "CDMX",        "Básico"),
    (5, "Sofía",  "Puebla",      "Básico"),
    (6, "Jorge",  "Jalisco",     "Premium"),
    (7, "Elena",  "CDMX",        "Premium"),
    (8, "Diego",  "Querétaro",   "Básico"),
    (9, "Laura",  "Puebla",      "Premium"),
    (10, "Pedro", "CDMX",        "Premium"),
    (11, "Andrea","Jalisco",     "Básico"),
    (12, "Miguel", "Veracruz",   "Básico"),
]

Definimos el schema:

In [3]:
clientes_schema = StructType([
    StructField("id_cliente", IntegerType(), False),
    StructField("nombre", StringType(), False),
    StructField("estado", StringType(), False),
    StructField("tipo_cliente", StringType(), False)
])

Creamos el DataFrame:

In [4]:
df_clientes = spark.createDataFrame(
    clientes_data,
    schema=clientes_schema
)

df_clientes.show(truncate=False)

+----------+------+----------+------------+
|id_cliente|nombre|estado    |tipo_cliente|
+----------+------+----------+------------+
|1         |Ana   |CDMX      |Premium     |
|2         |Luis  |Jalisco   |Básico      |
|3         |Marta |Nuevo León|Premium     |
|4         |Carlos|CDMX      |Básico      |
|5         |Sofía |Puebla    |Básico      |
|6         |Jorge |Jalisco   |Premium     |
|7         |Elena |CDMX      |Premium     |
|8         |Diego |Querétaro |Básico      |
|9         |Laura |Puebla    |Premium     |
|10        |Pedro |CDMX      |Premium     |
|11        |Andrea|Jalisco   |Básico      |
|12        |Miguel|Veracruz  |Básico      |
+----------+------+----------+------------+



### 2.2 DataFrame de transacciones

Cada fila representará una compra realizada por un cliente.

In [5]:
transacciones_data = [
    (1001, 1,  "2026-01-05", 1200.0, "Electrónica"),
    (1002, 1,  "2026-02-18",  850.0, "Hogar"),
    (1003, 1,  "2026-05-09", 2200.0, "Electrónica"),

    (1004, 2,  "2026-01-22",  450.0, "Ropa"),
    (1005, 2,  "2026-04-11",  780.0, "Hogar"),

    (1006, 3,  "2026-02-03", 3200.0, "Electrónica"),
    (1007, 3,  "2026-03-17", 1800.0, "Ropa"),
    (1008, 3,  "2026-06-21", 4100.0, "Electrónica"),

    (1009, 4,  "2026-03-04",  350.0, "Hogar"),

    (1010, 5,  "2026-01-30",  650.0, "Ropa"),
    (1011, 5,  "2026-05-16",  900.0, "Ropa"),

    (1012, 6,  "2026-02-27", 2700.0, "Electrónica"),
    (1013, 6,  "2026-07-01", 1950.0, "Hogar"),

    (1014, 7,  "2026-04-08", 1450.0, "Electrónica"),
    (1015, 7,  "2026-06-12", 2300.0, "Electrónica"),

    (1016, 8,  "2026-03-26",  520.0, "Ropa"),

    (1017, 9,  "2026-02-14", 1300.0, "Hogar"),
    (1018, 9,  "2026-05-28", 1700.0, "Electrónica"),

    (1019, 10, "2026-01-12", 2900.0, "Electrónica"),
    (1020, 10, "2026-03-20", 1100.0, "Hogar"),
    (1021, 10, "2026-07-15", 3500.0, "Electrónica"),

    # Transacción cuyo cliente no existe en df_clientes
    (1022, 99, "2026-06-30", 5000.0, "Electrónica"),
]

El registro con:

```text
id_cliente = 99
```

es intencional.

Nos servirá posteriormente para observar qué ocurre con diferentes tipos de join.

Definimos el schema:

In [6]:
transacciones_schema = StructType([
    StructField("id_transaccion", IntegerType(), False),
    StructField("id_cliente", IntegerType(), False),
    StructField("fecha", StringType(), False),
    StructField("monto", DoubleType(), False),
    StructField("categoria", StringType(), False)
])

Creamos el DataFrame y convertimos la fecha:

In [7]:
df_transacciones = (
    spark.createDataFrame(
        transacciones_data,
        schema=transacciones_schema
    )
    .withColumn(
        "fecha",
        F.to_date("fecha", "yyyy-MM-dd")
    )
)

df_transacciones.show(
    truncate=False
)

+--------------+----------+----------+------+-----------+
|id_transaccion|id_cliente|fecha     |monto |categoria  |
+--------------+----------+----------+------+-----------+
|1001          |1         |2026-01-05|1200.0|Electrónica|
|1002          |1         |2026-02-18|850.0 |Hogar      |
|1003          |1         |2026-05-09|2200.0|Electrónica|
|1004          |2         |2026-01-22|450.0 |Ropa       |
|1005          |2         |2026-04-11|780.0 |Hogar      |
|1006          |3         |2026-02-03|3200.0|Electrónica|
|1007          |3         |2026-03-17|1800.0|Ropa       |
|1008          |3         |2026-06-21|4100.0|Electrónica|
|1009          |4         |2026-03-04|350.0 |Hogar      |
|1010          |5         |2026-01-30|650.0 |Ropa       |
|1011          |5         |2026-05-16|900.0 |Ropa       |
|1012          |6         |2026-02-27|2700.0|Electrónica|
|1013          |6         |2026-07-01|1950.0|Hogar      |
|1014          |7         |2026-04-08|1450.0|Electrónica|
|1015         

Revisamos el schema:

In [8]:
df_transacciones.printSchema()

root
 |-- id_transaccion: integer (nullable = false)
 |-- id_cliente: integer (nullable = false)
 |-- fecha: date (nullable = false)
 |-- monto: double (nullable = false)
 |-- categoria: string (nullable = false)



## 3. Agregaciones simples

Hasta ahora hemos trabajado principalmente a nivel de registro.

Por ejemplo:

```text
Transacción 1001 → 1200
Transacción 1002 → 850
Transacción 1003 → 2200
```

Las agregaciones nos permiten resumir esos registros.

Por ejemplo:

```text
Cliente 1
   │
   ├── 1200
   ├── 850
   └── 2200
        │
        ▼
Total = 4250
```

### 3.1 Funciones de agregación

Spark proporciona funciones como:

```text
count()
sum()
avg()
min()
max()
```

Podemos utilizarlas directamente sobre todo el DataFrame.


In [9]:
df_transacciones.select(
    F.count("*").alias("num_transacciones"),
    F.sum("monto").alias("monto_total"),
    F.avg("monto").alias("monto_promedio"),
    F.min("monto").alias("monto_minimo"),
    F.max("monto").alias("monto_maximo")
).show()

+-----------------+-----------+-----------------+------------+------------+
|num_transacciones|monto_total|   monto_promedio|monto_minimo|monto_maximo|
+-----------------+-----------+-----------------+------------+------------+
|               22|    40900.0|1859.090909090909|       350.0|      5000.0|
+-----------------+-----------+-----------------+------------+------------+



Aquí estamos obteniendo estadísticas globales.

Es decir:

```text
Todas las transacciones
        ↓
    agregación
        ↓
una única fila resumen
```

## 4. Agrupaciones con `groupBy()`

La mayoría de las veces queremos calcular estadísticas **por grupo**.

Por ejemplo:

```text
CDMX
Jalisco
Puebla
...

o

Electrónica
Ropa
Hogar
```

Para ello utilizamos:

```python
groupBy()
```

### 4.1 Número de transacciones por categoría

In [10]:
df_transacciones.groupBy(
    "categoria"
).count().show()

+-----------+-----+
|  categoria|count|
+-----------+-----+
|Electrónica|   11|
|      Hogar|    6|
|       Ropa|    5|
+-----------+-----+



Conceptualmente:

```text
Transacciones
     │
     ▼
groupBy("categoria")
     │
     ├── Electrónica
     ├── Hogar
     └── Ropa
```

y posteriormente:

```text
cada grupo
   ↓
count()
```

## 5. `groupBy()` + `agg()`

`agg()` nos permite aplicar múltiples agregaciones sobre cada grupo.

In [11]:
resumen_categoria = (
    df_transacciones
    .groupBy("categoria")
    .agg(
        F.count("*").alias("num_transacciones"),
        F.sum("monto").alias("monto_total"),
        F.avg("monto").alias("monto_promedio"),
        F.min("monto").alias("monto_minimo"),
        F.max("monto").alias("monto_maximo")
    )
)

resumen_categoria.show()

+-----------+-----------------+-----------+--------------+------------+------------+
|  categoria|num_transacciones|monto_total|monto_promedio|monto_minimo|monto_maximo|
+-----------+-----------------+-----------+--------------+------------+------------+
|Electrónica|               11|    30250.0|        2750.0|      1200.0|      5000.0|
|      Hogar|                6|     6330.0|        1055.0|       350.0|      1950.0|
|       Ropa|                5|     4320.0|         864.0|       450.0|      1800.0|
+-----------+-----------------+-----------+--------------+------------+------------+



Este patrón:

```python
df.groupBy(...)
  .agg(...)
```

será extremadamente habitual en Spark.

### 5.1 Redondear resultados

Podemos combinar agregaciones con otras funciones:

In [12]:
resumen_categoria = (
    df_transacciones
    .groupBy("categoria")
    .agg(
        F.count("*").alias("num_transacciones"),
        F.round(
            F.sum("monto"),
            2
        ).alias("monto_total"),
        F.round(
            F.avg("monto"),
            2
        ).alias("monto_promedio")
    )
    .orderBy(
        F.col("monto_total").desc()
    )
)

resumen_categoria.show()

+-----------+-----------------+-----------+--------------+
|  categoria|num_transacciones|monto_total|monto_promedio|
+-----------+-----------------+-----------+--------------+
|Electrónica|               11|    30250.0|        2750.0|
|      Hogar|                6|     6330.0|        1055.0|
|       Ropa|                5|     4320.0|         864.0|
+-----------+-----------------+-----------+--------------+



## 6. Agrupar por múltiples columnas

También podemos utilizar más de una variable como clave de agrupación.

Por ejemplo:

In [13]:
resumen_categoria_mes = (
    df_transacciones
    .withColumn(
        "mes",
        F.month("fecha")
    )
    .groupBy(
        "mes",
        "categoria"
    )
    .agg(
        F.count("*").alias("num_transacciones"),
        F.sum("monto").alias("monto_total")
    )
    .orderBy(
        "mes",
        "categoria"
    )
)

resumen_categoria_mes.show()

+---+-----------+-----------------+-----------+
|mes|  categoria|num_transacciones|monto_total|
+---+-----------+-----------------+-----------+
|  1|Electrónica|                2|     4100.0|
|  1|       Ropa|                2|     1100.0|
|  2|Electrónica|                2|     5900.0|
|  2|      Hogar|                2|     2150.0|
|  3|      Hogar|                2|     1450.0|
|  3|       Ropa|                2|     2320.0|
|  4|Electrónica|                1|     1450.0|
|  4|      Hogar|                1|      780.0|
|  5|Electrónica|                2|     3900.0|
|  5|       Ropa|                1|      900.0|
|  6|Electrónica|                3|    11400.0|
|  7|Electrónica|                1|     3500.0|
|  7|      Hogar|                1|     1950.0|
+---+-----------+-----------------+-----------+



Ahora nuestros grupos se definen mediante la combinación:

```text
mes + categoria
```

Por ejemplo:

```text
1 + Electrónica
1 + Hogar
1 + Ropa

2 + Electrónica
2 + Hogar
...
```


# 7. Agregaciones por cliente

Podemos resumir las transacciones de cada cliente.

In [14]:
resumen_cliente = (
    df_transacciones
    .groupBy("id_cliente")
    .agg(
        F.count("*").alias("num_transacciones"),
        F.sum("monto").alias("gasto_total"),
        F.avg("monto").alias("ticket_promedio"),
        F.max("monto").alias("compra_maxima")
    )
    .orderBy(
        F.col("gasto_total").desc()
    )
)

resumen_cliente.show()

+----------+-----------------+-----------+------------------+-------------+
|id_cliente|num_transacciones|gasto_total|   ticket_promedio|compra_maxima|
+----------+-----------------+-----------+------------------+-------------+
|         3|                3|     9100.0|3033.3333333333335|       4100.0|
|        10|                3|     7500.0|            2500.0|       3500.0|
|        99|                1|     5000.0|            5000.0|       5000.0|
|         6|                2|     4650.0|            2325.0|       2700.0|
|         1|                3|     4250.0|1416.6666666666667|       2200.0|
|         7|                2|     3750.0|            1875.0|       2300.0|
|         9|                2|     3000.0|            1500.0|       1700.0|
|         5|                2|     1550.0|             775.0|        900.0|
|         2|                2|     1230.0|             615.0|        780.0|
|         8|                1|      520.0|             520.0|        520.0|
|         4|

Este DataFrame ya empieza a parecerse a una tabla de **features por cliente**:

```text
id_cliente
num_transacciones
gasto_total
ticket_promedio
compra_maxima
```

Este tipo de transformación es muy común antes de construir modelos de Machine Learning.

## 8. Contar valores distintos

En ocasiones no queremos contar filas, sino valores únicos.

Para ello podemos utilizar:

```python
F.countDistinct()
```

Por ejemplo, cuántas categorías diferentes ha comprado cada cliente:


In [15]:
df_transacciones.groupBy(
    "id_cliente"
).agg(
    F.count("*").alias("num_transacciones"),
    F.countDistinct(
        "categoria"
    ).alias("categorias_distintas")
).show()

+----------+-----------------+--------------------+
|id_cliente|num_transacciones|categorias_distintas|
+----------+-----------------+--------------------+
|         1|                3|                   2|
|         6|                2|                   2|
|         3|                3|                   2|
|         5|                2|                   1|
|         9|                2|                   2|
|         4|                1|                   1|
|         8|                1|                   1|
|         7|                2|                   1|
|        10|                3|                   2|
|         2|                2|                   2|
|        99|                1|                   1|
+----------+-----------------+--------------------+



## 9. Agregaciones condicionales

También podemos construir métricas que dependan de una condición.

Por ejemplo, queremos conocer para cada cliente:

```text
número de compras de Electrónica
```

Podemos hacerlo combinando:

```python
when()
```

con:

```python
sum()
```


In [16]:
resumen_condicional = (
    df_transacciones
    .groupBy("id_cliente")
    .agg(
        F.count("*").alias(
            "num_transacciones"
        ),

        F.sum(
            F.when(
                F.col("categoria") == "Electrónica",
                1
            ).otherwise(0)
        ).alias(
            "compras_electronica"
        ),

        F.sum(
            F.when(
                F.col("categoria") == "Electrónica",
                F.col("monto")
            ).otherwise(0)
        ).alias(
            "gasto_electronica"
        )
    )
)

resumen_condicional.show()

+----------+-----------------+-------------------+-----------------+
|id_cliente|num_transacciones|compras_electronica|gasto_electronica|
+----------+-----------------+-------------------+-----------------+
|         1|                3|                  2|           3400.0|
|         2|                2|                  0|              0.0|
|         3|                3|                  2|           7300.0|
|         4|                1|                  0|              0.0|
|         5|                2|                  0|              0.0|
|         6|                2|                  1|           2700.0|
|         7|                2|                  2|           3750.0|
|         8|                1|                  0|              0.0|
|         9|                2|                  1|           1700.0|
|        10|                3|                  2|           6400.0|
|        99|                1|                  1|           5000.0|
+----------+-----------------+----

Este patrón es muy útil para construir variables derivadas.

### 9.1 Porcentaje de compras de Electrónica

Podemos continuar la transformación:

In [17]:
resumen_condicional = (
    resumen_condicional
    .withColumn(
        "pct_compras_electronica",
        F.round(
            F.col("compras_electronica")
            / F.col("num_transacciones")
            * 100,
            2
        )
    )
)

resumen_condicional.show()

+----------+-----------------+-------------------+-----------------+-----------------------+
|id_cliente|num_transacciones|compras_electronica|gasto_electronica|pct_compras_electronica|
+----------+-----------------+-------------------+-----------------+-----------------------+
|         1|                3|                  2|           3400.0|                  66.67|
|         2|                2|                  0|              0.0|                    0.0|
|         3|                3|                  2|           7300.0|                  66.67|
|         4|                1|                  0|              0.0|                    0.0|
|         5|                2|                  0|              0.0|                    0.0|
|         6|                2|                  1|           2700.0|                   50.0|
|         7|                2|                  2|           3750.0|                  100.0|
|         8|                1|                  0|              0.0|  

Por ejemplo:

```text
Cliente
    │
    ├── 4 compras
    ├── 3 electrónica
    │
    ▼
75 % electrónica
```

## 10. Primera conexión con el modelo de ejecución

Recordemos que `groupBy()` suele requerir reunir los registros que pertenecen a la misma clave.

Por ejemplo:

In [18]:
resumen_cliente.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Exchange (4)
            +- HashAggregate (3)
               +- Project (2)
                  +- Scan ExistingRDD (1)


(1) Scan ExistingRDD
Output [5]: [id_transaccion#17, id_cliente#18, fecha#19, monto#20, categoria#21]
Arguments: [id_transaccion#17, id_cliente#18, fecha#19, monto#20, categoria#21], MapPartitionsRDD[11] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Project
Output [2]: [id_cliente#18, monto#20]
Input [5]: [id_transaccion#17, id_cliente#18, fecha#19, monto#20, categoria#21]

(3) HashAggregate
Input [2]: [id_cliente#18, monto#20]
Keys [1]: [id_cliente#18]
Functions [4]: [partial_count(1), partial_sum(monto#20), partial_avg(monto#20), partial_max(monto#20)]
Aggregate Attributes [5]: [count#231L, sum#232, sum#233, count#234L, max#235]
Results [6]: [id_cliente#18, count#236L, sum#237, sum#238, count#239L, m

Busca nuevamente elementos similares a:

```text
Exchange
HashAggregate
```

De forma conceptual:

```text
PARTICIONES ORIGINALES

P0
cliente 1
cliente 2
cliente 3

P1
cliente 1
cliente 4
cliente 5

P2
cliente 2
cliente 3
cliente 5

        ↓

      SHUFFLE

        ↓

datos redistribuidos
por id_cliente

        ↓

agregación
```

Aquí estamos aplicando directamente los conceptos estudiados en `02_01`.

## 11. Un problema importante: `groupBy()` elimina el detalle

Supongamos que tenemos:

```text
cliente | monto
--------|------
1       | 1200
1       | 850
1       | 2200
```

Al ejecutar:

```python
df.groupBy(
    "id_cliente"
).agg(
    F.sum("monto")
)
```

obtenemos:

```text
cliente | gasto_total
--------|------------
1       | 4250
```

Las filas individuales desaparecen.

Esto es exactamente lo que queremos cuando buscamos un resumen.

Pero existen situaciones donde queremos calcular una métrica por cliente **sin perder las transacciones originales**.

Por ejemplo:

```text
transacción | cliente | monto | gasto_total_cliente
------------|---------|-------|--------------------
1001        | 1       | 1200  | 4250
1002        | 1       | 850   | 4250
1003        | 1       | 2200  | 4250
```

Para resolver este tipo de problema utilizaremos más adelante las:

```text
Window Functions
```

Esta diferencia será fundamental:

```text
groupBy()
    ↓
reduce filas


Window
    ↓
mantiene filas
+ agrega contexto del grupo
```

## 12. Resumen de agregaciones

Hasta ahora hemos trabajado con:

```text
count()
sum()
avg()
min()
max()
countDistinct()
```

y las hemos combinado mediante:

```python
groupBy(...).agg(...)
```

Además, aprendimos a construir agregaciones condicionales utilizando:

```text
sum(
    when(...)
)
```

La idea principal es:

> `groupBy()` transforma múltiples registros pertenecientes a un grupo en uno o varios valores resumen.

En el siguiente bloque relacionaremos nuestros dos DataFrames:

```text
df_clientes
      +
df_transacciones
      ↓
     JOIN
```

y estudiaremos:

```text
inner
left
right
full
left_semi
left_anti
```

además de uno de los problemas más frecuentes al trabajar con joins en Spark: **columnas duplicadas y referencias ambiguas**.

## 13. Joins entre DataFrames

Los **joins** permiten combinar información proveniente de diferentes DataFrames utilizando una o varias columnas relacionadas.

En nuestro caso tenemos:

```text
df_clientes
    │
    │ id_cliente
    │
    ▼
df_transacciones
```

La columna común es:

```text
id_cliente
```

Por lo tanto, podremos relacionar cada transacción con la información del cliente correspondiente.

## 14. `inner join`

El `inner join` conserva únicamente los registros cuya clave existe en ambos DataFrames.

In [19]:
df_inner = (
    df_clientes
    .join(
        df_transacciones,
        on="id_cliente",
        how="inner"
    )
)

df_inner.show(
    truncate=False
)

+----------+------+----------+------------+--------------+----------+------+-----------+
|id_cliente|nombre|estado    |tipo_cliente|id_transaccion|fecha     |monto |categoria  |
+----------+------+----------+------------+--------------+----------+------+-----------+
|1         |Ana   |CDMX      |Premium     |1001          |2026-01-05|1200.0|Electrónica|
|1         |Ana   |CDMX      |Premium     |1002          |2026-02-18|850.0 |Hogar      |
|1         |Ana   |CDMX      |Premium     |1003          |2026-05-09|2200.0|Electrónica|
|2         |Luis  |Jalisco   |Básico      |1004          |2026-01-22|450.0 |Ropa       |
|2         |Luis  |Jalisco   |Básico      |1005          |2026-04-11|780.0 |Hogar      |
|3         |Marta |Nuevo León|Premium     |1006          |2026-02-03|3200.0|Electrónica|
|3         |Marta |Nuevo León|Premium     |1007          |2026-03-17|1800.0|Ropa       |
|3         |Marta |Nuevo León|Premium     |1008          |2026-06-21|4100.0|Electrónica|
|4         |Carlos|CD

Conceptualmente:

```text
CLIENTES                 TRANSACCIONES

1 Ana                    1 compra A
2 Luis                   1 compra B
3 Marta                  3 compra C
12 Miguel
                         99 compra X

             INNER
               ↓

solo claves presentes
en ambos lados
```

Por lo tanto:

- `id_cliente = 12` no aparece si no tiene transacciones;
- `id_cliente = 99` tampoco aparece porque no existe en `df_clientes`.

### 14.1 Inspeccionar el resultado

In [20]:
print(
    "Número de registros:",
    df_inner.count()
)

df_inner.printSchema()

Número de registros: 21
root
 |-- id_cliente: integer (nullable = false)
 |-- nombre: string (nullable = false)
 |-- estado: string (nullable = false)
 |-- tipo_cliente: string (nullable = false)
 |-- id_transaccion: integer (nullable = false)
 |-- fecha: date (nullable = false)
 |-- monto: double (nullable = false)
 |-- categoria: string (nullable = false)



Observa que la relación:

```text
1 cliente
    ↓
muchas transacciones
```

provoca que los datos del cliente se repitan en cada una de sus transacciones.

Por ejemplo:

```text
Ana | CDMX | Premium | compra 1001
Ana | CDMX | Premium | compra 1002
Ana | CDMX | Premium | compra 1003
```

Esto es completamente normal en una relación uno-a-muchos.


## 15. `left join`

El `left join` conserva **todos los registros del DataFrame izquierdo**, exista o no una coincidencia en el DataFrame derecho.

En este caso:

In [25]:
df_left = (
    df_clientes
    .join(
        df_transacciones,
        on="id_cliente",
        how="left"
    )
)

df_left.show( 30,
    truncate=False
)

+----------+------+----------+------------+--------------+----------+------+-----------+
|id_cliente|nombre|estado    |tipo_cliente|id_transaccion|fecha     |monto |categoria  |
+----------+------+----------+------------+--------------+----------+------+-----------+
|1         |Ana   |CDMX      |Premium     |1003          |2026-05-09|2200.0|Electrónica|
|1         |Ana   |CDMX      |Premium     |1002          |2026-02-18|850.0 |Hogar      |
|1         |Ana   |CDMX      |Premium     |1001          |2026-01-05|1200.0|Electrónica|
|2         |Luis  |Jalisco   |Básico      |1005          |2026-04-11|780.0 |Hogar      |
|2         |Luis  |Jalisco   |Básico      |1004          |2026-01-22|450.0 |Ropa       |
|3         |Marta |Nuevo León|Premium     |1008          |2026-06-21|4100.0|Electrónica|
|3         |Marta |Nuevo León|Premium     |1007          |2026-03-17|1800.0|Ropa       |
|3         |Marta |Nuevo León|Premium     |1006          |2026-02-03|3200.0|Electrónica|
|4         |Carlos|CD

Conceptualmente:

```text
df_clientes
    ↓
TODO se conserva

df_transacciones
    ↓
se agrega si existe coincidencia
```

Por lo tanto, un cliente sin transacciones permanecerá en el resultado.

Las columnas provenientes de `df_transacciones` tendrán:

```text
null
```

cuando no exista ninguna coincidencia.

### 15.1 Identificar clientes sin transacciones

Podemos aprovechar el resultado del `left join`:

In [22]:
df_left.filter(
    F.col("id_transaccion").isNull()
).show(
    truncate=False
)

+----------+------+--------+------------+--------------+-----+-----+---------+
|id_cliente|nombre|estado  |tipo_cliente|id_transaccion|fecha|monto|categoria|
+----------+------+--------+------------+--------------+-----+-----+---------+
|11        |Andrea|Jalisco |Básico      |NULL          |NULL |NULL |NULL     |
|12        |Miguel|Veracruz|Básico      |NULL          |NULL |NULL |NULL     |
+----------+------+--------+------------+--------------+-----+-----+---------+



Con nuestros datos debería aparecer el cliente que no tiene ninguna transacción registrada.

## 16. `right join`

El `right join` hace lo contrario: conserva todos los registros del DataFrame derecho.


In [24]:
df_right = (
    df_clientes
    .join(
        df_transacciones,
        on="id_cliente",
        how="right"
    )
)

df_right.show(30,
    truncate=False
)

+----------+------+----------+------------+--------------+----------+------+-----------+
|id_cliente|nombre|estado    |tipo_cliente|id_transaccion|fecha     |monto |categoria  |
+----------+------+----------+------------+--------------+----------+------+-----------+
|1         |Ana   |CDMX      |Premium     |1001          |2026-01-05|1200.0|Electrónica|
|1         |Ana   |CDMX      |Premium     |1002          |2026-02-18|850.0 |Hogar      |
|1         |Ana   |CDMX      |Premium     |1003          |2026-05-09|2200.0|Electrónica|
|2         |Luis  |Jalisco   |Básico      |1004          |2026-01-22|450.0 |Ropa       |
|2         |Luis  |Jalisco   |Básico      |1005          |2026-04-11|780.0 |Hogar      |
|3         |Marta |Nuevo León|Premium     |1006          |2026-02-03|3200.0|Electrónica|
|3         |Marta |Nuevo León|Premium     |1007          |2026-03-17|1800.0|Ropa       |
|3         |Marta |Nuevo León|Premium     |1008          |2026-06-21|4100.0|Electrónica|
|4         |Carlos|CD

En este caso:

```text
df_transacciones
       ↓
TODO se conserva
```

Por eso también debería aparecer:

```text
id_cliente = 99
```

aunque no exista en `df_clientes`.

Las columnas:

```text
nombre
estado
tipo_cliente
```

serán `null` para esa transacción.

## 17. `full join`

Un `full join` conserva todos los registros de ambos DataFrames.

In [27]:
df_full = (
    df_clientes
    .join(
        df_transacciones,
        on="id_cliente",
        how="full"
    )
)

df_full.show(40,
    truncate=False
)

+----------+------+----------+------------+--------------+----------+------+-----------+
|id_cliente|nombre|estado    |tipo_cliente|id_transaccion|fecha     |monto |categoria  |
+----------+------+----------+------------+--------------+----------+------+-----------+
|1         |Ana   |CDMX      |Premium     |1001          |2026-01-05|1200.0|Electrónica|
|1         |Ana   |CDMX      |Premium     |1002          |2026-02-18|850.0 |Hogar      |
|1         |Ana   |CDMX      |Premium     |1003          |2026-05-09|2200.0|Electrónica|
|2         |Luis  |Jalisco   |Básico      |1004          |2026-01-22|450.0 |Ropa       |
|2         |Luis  |Jalisco   |Básico      |1005          |2026-04-11|780.0 |Hogar      |
|3         |Marta |Nuevo León|Premium     |1006          |2026-02-03|3200.0|Electrónica|
|3         |Marta |Nuevo León|Premium     |1007          |2026-03-17|1800.0|Ropa       |
|3         |Marta |Nuevo León|Premium     |1008          |2026-06-21|4100.0|Electrónica|
|4         |Carlos|CD

Aquí aparecerán:

- clientes con transacciones;
- clientes sin transacciones;
- transacciones cuyo cliente no existe.

Conceptualmente:

```text
LEFT
  +
RIGHT
  ↓
FULL
```

## 18. Comparación de los principales joins

Podemos resumirlos de esta forma:

| Join | Conserva izquierda | Conserva derecha | Solo coincidencias |
|---|---:|---:|---:|
| `inner` | No | No | Sí |
| `left` | Sí | No | No |
| `right` | No | Sí | No |
| `full` | Sí | Sí | No |

Otra forma de pensarlo:

```text
INNER
    ↓
intersección


LEFT
    ↓
todo izquierda
+ coincidencias derecha


RIGHT
    ↓
todo derecha
+ coincidencias izquierda


FULL
    ↓
unión de ambos lados
```


## 19. `left_semi`

El `left_semi` es especialmente interesante.

Devuelve únicamente filas del DataFrame izquierdo que **sí tienen al menos una coincidencia** en el derecho.

In [28]:
df_clientes_con_compras = (
    df_clientes
    .join(
        df_transacciones,
        on="id_cliente",
        how="left_semi"
    )
)

df_clientes_con_compras.show(
    truncate=False
)

+----------+------+----------+------------+
|id_cliente|nombre|estado    |tipo_cliente|
+----------+------+----------+------------+
|1         |Ana   |CDMX      |Premium     |
|2         |Luis  |Jalisco   |Básico      |
|3         |Marta |Nuevo León|Premium     |
|4         |Carlos|CDMX      |Básico      |
|5         |Sofía |Puebla    |Básico      |
|6         |Jorge |Jalisco   |Premium     |
|7         |Elena |CDMX      |Premium     |
|8         |Diego |Querétaro |Básico      |
|9         |Laura |Puebla    |Premium     |
|10        |Pedro |CDMX      |Premium     |
+----------+------+----------+------------+



La diferencia con un `inner join` es muy importante.

El resultado contiene únicamente columnas de:

```text
df_clientes
```

No incorpora las columnas de las transacciones.

Conceptualmente:

```text
¿Este cliente aparece
en transacciones?

Sí
 ↓
conservar cliente

No
 ↓
descartar cliente
```

Esto se parece a una operación SQL:

```sql
WHERE EXISTS (...)
```

## 20. `left_anti`

El `left_anti` hace exactamente lo contrario.

Devuelve las filas del DataFrame izquierdo que **no tienen coincidencia** en el DataFrame derecho.

In [29]:
df_clientes_sin_compras = (
    df_clientes
    .join(
        df_transacciones,
        on="id_cliente",
        how="left_anti"
    )
)

df_clientes_sin_compras.show(
    truncate=False
)

+----------+------+--------+------------+
|id_cliente|nombre|estado  |tipo_cliente|
+----------+------+--------+------------+
|11        |Andrea|Jalisco |Básico      |
|12        |Miguel|Veracruz|Básico      |
+----------+------+--------+------------+



Este tipo de join es extremadamente útil para detectar:

- clientes sin actividad;
- registros sin correspondencia;
- datos faltantes entre fuentes;
- diferencias entre datasets.

Conceptualmente:

```text
df_clientes
     │
     ▼
¿existe en transacciones?
     │
   NO
     │
     ▼
conservar
```

## 21. Detectar transacciones sin cliente

Podemos invertir el análisis.

Queremos identificar transacciones cuyo `id_cliente` no aparezca en la tabla de clientes.

In [30]:
df_transacciones_sin_cliente = (
    df_transacciones
    .join(
        df_clientes,
        on="id_cliente",
        how="left_anti"
    )
)

df_transacciones_sin_cliente.show(
    truncate=False
)

+----------+--------------+----------+------+-----------+
|id_cliente|id_transaccion|fecha     |monto |categoria  |
+----------+--------------+----------+------+-----------+
|99        |1022          |2026-06-30|5000.0|Electrónica|
+----------+--------------+----------+------+-----------+



Aquí debería aparecer:

```text
id_cliente = 99
```

Este patrón es muy útil para realizar controles de calidad de datos

## 22. Joins con expresiones

Hasta ahora hemos utilizado:

```python
on="id_cliente"
```

porque la columna tiene exactamente el mismo nombre en ambos DataFrames.

También podemos escribir explícitamente la condición.

Para ello es recomendable utilizar aliases:

In [31]:
clientes = df_clientes.alias("c")
transacciones = df_transacciones.alias("t")

Después:

In [32]:
df_join_explicito = (
    clientes
    .join(
        transacciones,
        F.col("c.id_cliente")
        == F.col("t.id_cliente"),
        "inner"
    )
)

df_join_explicito.show(
    truncate=False
)

+----------+------+----------+------------+--------------+----------+----------+------+-----------+
|id_cliente|nombre|estado    |tipo_cliente|id_transaccion|id_cliente|fecha     |monto |categoria  |
+----------+------+----------+------------+--------------+----------+----------+------+-----------+
|1         |Ana   |CDMX      |Premium     |1001          |1         |2026-01-05|1200.0|Electrónica|
|1         |Ana   |CDMX      |Premium     |1002          |1         |2026-02-18|850.0 |Hogar      |
|1         |Ana   |CDMX      |Premium     |1003          |1         |2026-05-09|2200.0|Electrónica|
|2         |Luis  |Jalisco   |Básico      |1004          |2         |2026-01-22|450.0 |Ropa       |
|2         |Luis  |Jalisco   |Básico      |1005          |2         |2026-04-11|780.0 |Hogar      |
|3         |Marta |Nuevo León|Premium     |1006          |3         |2026-02-03|3200.0|Electrónica|
|3         |Marta |Nuevo León|Premium     |1007          |3         |2026-03-17|1800.0|Ropa       |


Este estilo se vuelve especialmente importante cuando las claves tienen nombres distintos o cuando necesitamos condiciones más complejas.

## 23. Columnas duplicadas y ambigüedad

Cuando utilizamos:

```python
on="id_cliente"
```

Spark reconoce que esa columna es la clave común y mantiene una sola columna `id_cliente`.

Pero si escribimos una expresión:


In [33]:
F.col("c.id_cliente") == F.col("t.id_cliente")

Column<'=(c.id_cliente, t.id_cliente)'>

ambas columnas pueden quedar presentes en el resultado.

Podemos comprobar:

In [34]:
print(
    df_join_explicito.columns
)

['id_cliente', 'nombre', 'estado', 'tipo_cliente', 'id_transaccion', 'id_cliente', 'fecha', 'monto', 'categoria']


Podríamos encontrar dos columnas con nombre:

```text
id_cliente
```

Esto puede provocar problemas posteriores.

Por ejemplo, una referencia como:

```python
F.col("id_cliente")
```

puede resultar ambigua porque Spark no sabe cuál queremos utilizar.

## 24. Resolver columnas ambiguas con aliases

Una práctica recomendable consiste en seleccionar explícitamente las columnas después del join.

In [35]:
df_join_limpio = (
    clientes
    .join(
        transacciones,
        F.col("c.id_cliente")
        == F.col("t.id_cliente"),
        "inner"
    )
    .select(
        F.col("c.id_cliente").alias("id_cliente"),
        F.col("c.nombre"),
        F.col("c.estado"),
        F.col("c.tipo_cliente"),
        F.col("t.id_transaccion"),
        F.col("t.fecha"),
        F.col("t.monto"),
        F.col("t.categoria")
    )
)

df_join_limpio.show(
    truncate=False
)

+----------+------+----------+------------+--------------+----------+------+-----------+
|id_cliente|nombre|estado    |tipo_cliente|id_transaccion|fecha     |monto |categoria  |
+----------+------+----------+------------+--------------+----------+------+-----------+
|1         |Ana   |CDMX      |Premium     |1001          |2026-01-05|1200.0|Electrónica|
|1         |Ana   |CDMX      |Premium     |1002          |2026-02-18|850.0 |Hogar      |
|1         |Ana   |CDMX      |Premium     |1003          |2026-05-09|2200.0|Electrónica|
|2         |Luis  |Jalisco   |Básico      |1004          |2026-01-22|450.0 |Ropa       |
|2         |Luis  |Jalisco   |Básico      |1005          |2026-04-11|780.0 |Hogar      |
|3         |Marta |Nuevo León|Premium     |1006          |2026-02-03|3200.0|Electrónica|
|3         |Marta |Nuevo León|Premium     |1007          |2026-03-17|1800.0|Ropa       |
|3         |Marta |Nuevo León|Premium     |1008          |2026-06-21|4100.0|Electrónica|
|4         |Carlos|CD

Ahora nuestro resultado tiene una estructura clara y controlada.

Este patrón:

```python
df1.alias("a")
df2.alias("b")

a.join(b, ...)
 .select(...)
```

es muy recomendable en pipelines complejos.

## 25. Join + agregación

Ahora combinaremos dos conceptos del notebook.

Primero relacionaremos clientes y transacciones:

In [36]:
df_cliente_transacciones = (
    df_clientes
    .join(
        df_transacciones,
        on="id_cliente",
        how="left"
    )
)

Después construiremos métricas por cliente:

In [37]:
resumen_cliente_completo = (
    df_cliente_transacciones
    .groupBy(
        "id_cliente",
        "nombre",
        "estado",
        "tipo_cliente"
    )
    .agg(
        F.count(
            "id_transaccion"
        ).alias(
            "num_transacciones"
        ),

        F.round(
            F.sum("monto"),
            2
        ).alias(
            "gasto_total"
        ),

        F.round(
            F.avg("monto"),
            2
        ).alias(
            "ticket_promedio"
        ),

        F.max("monto").alias(
            "compra_maxima"
        )
    )
    .orderBy(
        F.col("gasto_total")
        .desc_nulls_last()
    )
)

resumen_cliente_completo.show(
    truncate=False
)

+----------+------+----------+------------+-----------------+-----------+---------------+-------------+
|id_cliente|nombre|estado    |tipo_cliente|num_transacciones|gasto_total|ticket_promedio|compra_maxima|
+----------+------+----------+------------+-----------------+-----------+---------------+-------------+
|3         |Marta |Nuevo León|Premium     |3                |9100.0     |3033.33        |4100.0       |
|10        |Pedro |CDMX      |Premium     |3                |7500.0     |2500.0         |3500.0       |
|6         |Jorge |Jalisco   |Premium     |2                |4650.0     |2325.0         |2700.0       |
|1         |Ana   |CDMX      |Premium     |3                |4250.0     |1416.67        |2200.0       |
|7         |Elena |CDMX      |Premium     |2                |3750.0     |1875.0         |2300.0       |
|9         |Laura |Puebla    |Premium     |2                |3000.0     |1500.0         |1700.0       |
|5         |Sofía |Puebla    |Básico      |2                |155

Ahora tenemos un DataFrame mucho más útil:

```text
id_cliente
nombre
estado
tipo_cliente
num_transacciones
gasto_total
ticket_promedio
compra_maxima
```

Este tipo de tabla podría utilizarse posteriormente para:

- segmentación;
- reporting;
- Machine Learning;
- análisis de comportamiento;
- scoring.

## 26. Una diferencia importante: `count("*")` vs `count(columna)`

En un `left join`, un cliente sin transacciones seguirá teniendo una fila.

Por eso:

```python
F.count("*")
```

podría devolver:

```text
1
```

aunque no exista ninguna transacción.

En cambio:

```python
F.count("id_transaccion")
```

solo cuenta valores no nulos.

Por ello, en este caso utilizamos:

```python
F.count("id_transaccion")
```

para obtener correctamente:

```text
0 transacciones
```

para clientes sin actividad.

Esta pequeña diferencia puede producir errores importantes en análisis reales.

## 27. Joins y shuffle

Los joins suelen requerir redistribuir datos para conseguir que registros con la misma clave puedan procesarse juntos.

Podemos inspeccionar:

In [38]:
df_inner.explain(
    "formatted"
)

== Physical Plan ==
AdaptiveSparkPlan (10)
+- Project (9)
   +- SortMergeJoin Inner (8)
      :- Sort (3)
      :  +- Exchange (2)
      :     +- Scan ExistingRDD (1)
      +- Sort (7)
         +- Exchange (6)
            +- Project (5)
               +- Scan ExistingRDD (4)


(1) Scan ExistingRDD
Output [4]: [id_cliente#0, nombre#1, estado#2, tipo_cliente#3]
Arguments: [id_cliente#0, nombre#1, estado#2, tipo_cliente#3], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Exchange
Input [4]: [id_cliente#0, nombre#1, estado#2, tipo_cliente#3]
Arguments: hashpartitioning(id_cliente#0, 200), ENSURE_REQUIREMENTS, [plan_id=2441]

(3) Sort
Input [4]: [id_cliente#0, nombre#1, estado#2, tipo_cliente#3]
Arguments: [id_cliente#0 ASC NULLS FIRST], false, 0

(4) Scan ExistingRDD
Output [5]: [id_transaccion#17, id_cliente#18, fecha#19, monto#20, categoria#21]
Arguments: [id_transaccion#17, id_cliente#18, fecha#19, monto#20, cate

Busca elementos como:

```text
Exchange
SortMergeJoin
```

o algún otro operador de join elegido por Spark.

El plan exacto puede variar porque Spark decide qué estrategia utilizar dependiendo de:

- tamaño de los DataFrames;
- estadísticas disponibles;
- configuración;
- Adaptive Query Execution;
- posibilidad de utilizar broadcast.

Conceptualmente:

```text
df_clientes
    │
    ├── repartición por id_cliente
    │
    ▼

       JOIN

    ▲
    │
    ├── repartición por id_cliente
    │
df_transacciones
```

Esto explica por qué los joins pueden ser operaciones costosas en Big Data.

## 28. ¿Todos los joins necesitan necesariamente un shuffle?

No.

Spark dispone de diferentes estrategias de join.

Una especialmente importante es:

```text
Broadcast Join
```

Si uno de los DataFrames es suficientemente pequeño, Spark puede enviar una copia de ese DataFrame a los ejecutores.

Conceptualmente:

```text
DataFrame pequeño
      │
      ├────────► Executor 1
      ├────────► Executor 2
      ├────────► Executor 3
      └────────► Executor 4

DataFrame grande
      ↓
permanece distribuido
```

Así se puede evitar redistribuir completamente el DataFrame grande.

No entraremos todavía en optimización manual de joins; lo estudiaremos en una fase posterior.

Por ahora basta con recordar:

> Un join puede implicar shuffle, pero Spark dispone de diferentes estrategias físicas para ejecutarlo.

## 29. Ejercicio práctico de joins

Construye un DataFrame llamado:

```python
df_clientes_resumen
```

que contenga **todos los clientes**, hayan comprado o no.

Debe incluir:

```text
id_cliente
nombre
estado
tipo_cliente
num_transacciones
gasto_total
```

Condiciones:

1. utilizar un `left join`;
2. contar correctamente las transacciones;
3. reemplazar `gasto_total` nulo por `0`;
4. ordenar de mayor a menor gasto;
5. conservar también clientes sin transacciones.


### 29.1 Solución

In [39]:
df_clientes_resumen = (
    df_clientes

    .join(
        df_transacciones,
        on="id_cliente",
        how="left"
    )

    .groupBy(
        "id_cliente",
        "nombre",
        "estado",
        "tipo_cliente"
    )

    .agg(
        F.count(
            "id_transaccion"
        ).alias(
            "num_transacciones"
        ),

        F.sum(
            "monto"
        ).alias(
            "gasto_total"
        )
    )

    .fillna({
        "gasto_total": 0.0
    })

    .orderBy(
        F.col(
            "gasto_total"
        ).desc()
    )
)

df_clientes_resumen.show(
    truncate=False
)

+----------+------+----------+------------+-----------------+-----------+
|id_cliente|nombre|estado    |tipo_cliente|num_transacciones|gasto_total|
+----------+------+----------+------------+-----------------+-----------+
|3         |Marta |Nuevo León|Premium     |3                |9100.0     |
|10        |Pedro |CDMX      |Premium     |3                |7500.0     |
|6         |Jorge |Jalisco   |Premium     |2                |4650.0     |
|1         |Ana   |CDMX      |Premium     |3                |4250.0     |
|7         |Elena |CDMX      |Premium     |2                |3750.0     |
|9         |Laura |Puebla    |Premium     |2                |3000.0     |
|5         |Sofía |Puebla    |Básico      |2                |1550.0     |
|2         |Luis  |Jalisco   |Básico      |2                |1230.0     |
|8         |Diego |Querétaro |Básico      |1                |520.0      |
|4         |Carlos|CDMX      |Básico      |1                |350.0      |
|11        |Andrea|Jalisco   |Básico  

Este ejercicio integra:

```text
join
 ↓
groupBy
 ↓
aggregation
 ↓
null handling
 ↓
sorting
```

y representa un patrón muy habitual en procesos ETL.


## 30. Resumen de joins

En esta sección utilizamos:

```text
inner
left
right
full
left_semi
left_anti
```

Una forma práctica de recordarlos es:

```text
INNER
→ dame únicamente coincidencias

LEFT
→ conserva todo el lado izquierdo

RIGHT
→ conserva todo el lado derecho

FULL
→ conserva ambos lados

LEFT SEMI
→ dame filas izquierdas que sí tienen match

LEFT ANTI
→ dame filas izquierdas que NO tienen match
```

También aprendimos dos prácticas importantes:

```text
alias()
   +
select explícito
```

para evitar referencias ambiguas después de joins complejos.

Finalmente vimos que los joins pueden introducir:

```text
shuffle
```

y que Spark puede elegir diferentes estrategias físicas para ejecutarlos.

## 31. Window Functions

Las **Window Functions** permiten calcular métricas sobre grupos de registros sin reducir el número de filas del DataFrame.

Esta es la diferencia principal respecto a `groupBy()`.

Con `groupBy()`:

```text
varias filas
    ↓
una fila resumen
```

Con una ventana:

```text
varias filas
    ↓
se conservan las filas
    +
se agrega información del grupo
```

Por ejemplo, podemos tener:

```text
transacción | cliente | monto | gasto_total_cliente
------------|---------|-------|--------------------
1001        | 1       | 1200  | 4250
1002        | 1       | 850   | 4250
1003        | 1       | 2200  | 4250
```

La información agregada del cliente aparece en cada transacción.

## 32. Definir una ventana

Las ventanas se construyen utilizando:

```python
Window.partitionBy(...)
```

Por ejemplo:

In [40]:
ventana_cliente = (
    Window
    .partitionBy("id_cliente")
)

Esto indica que los cálculos deben realizarse por cliente.

Podemos imaginarlo así:

```text
Cliente 1
 ├── transacción A
 ├── transacción B
 └── transacción C

Cliente 2
 ├── transacción D
 └── transacción E
```

Cada cliente forma una ventana independiente.


## 33. Agregaciones sin perder filas

Podemos calcular el gasto total por cliente sin utilizar `groupBy()`.

In [41]:
df_window = (
    df_transacciones
    .withColumn(
        "gasto_total_cliente",
        F.sum("monto").over(
            ventana_cliente
        )
    )
)

df_window.show(
    truncate=False
)

+--------------+----------+----------+------+-----------+-------------------+
|id_transaccion|id_cliente|fecha     |monto |categoria  |gasto_total_cliente|
+--------------+----------+----------+------+-----------+-------------------+
|1001          |1         |2026-01-05|1200.0|Electrónica|4250.0             |
|1002          |1         |2026-02-18|850.0 |Hogar      |4250.0             |
|1003          |1         |2026-05-09|2200.0|Electrónica|4250.0             |
|1004          |2         |2026-01-22|450.0 |Ropa       |1230.0             |
|1005          |2         |2026-04-11|780.0 |Hogar      |1230.0             |
|1006          |3         |2026-02-03|3200.0|Electrónica|9100.0             |
|1007          |3         |2026-03-17|1800.0|Ropa       |9100.0             |
|1008          |3         |2026-06-21|4100.0|Electrónica|9100.0             |
|1009          |4         |2026-03-04|350.0 |Hogar      |350.0              |
|1010          |5         |2026-01-30|650.0 |Ropa       |1550.0 

Observa que seguimos teniendo una fila por transacción.

### 33.1 Comparación con `groupBy()`

Con `groupBy()`:

```python
df_transacciones.groupBy(
    "id_cliente"
).agg(
    F.sum("monto")
)
```

obtenemos una fila por cliente.

Con Window:

```python
F.sum("monto").over(
    ventana_cliente
)
```

mantenemos todas las transacciones.

La elección depende de lo que necesitemos:

```text
quiero resumen
    ↓
groupBy()


quiero detalle + contexto del grupo
    ↓
Window
```

## 34. Ordenar dentro de una ventana

Podemos agregar un criterio de orden.

In [43]:
ventana_cliente_fecha = (
    Window
    .partitionBy("id_cliente")
    .orderBy("fecha")
)

Ahora Spark interpreta cada grupo en orden cronológico.

Conceptualmente:

```text
Cliente 1

2026-01-05
2026-02-18
2026-05-09
```

Esto permite construir rankings, acumulados y comparaciones entre registros consecutivos.

## 35. `row_number()`

`row_number()` asigna un número consecutivo dentro de cada ventana.

In [44]:
df_row_number = (
    df_transacciones
    .withColumn(
        "num_transaccion_cliente",
        F.row_number().over(
            ventana_cliente_fecha
        )
    )
)

df_row_number.show(
    truncate=False
)

+--------------+----------+----------+------+-----------+-----------------------+
|id_transaccion|id_cliente|fecha     |monto |categoria  |num_transaccion_cliente|
+--------------+----------+----------+------+-----------+-----------------------+
|1001          |1         |2026-01-05|1200.0|Electrónica|1                      |
|1002          |1         |2026-02-18|850.0 |Hogar      |2                      |
|1003          |1         |2026-05-09|2200.0|Electrónica|3                      |
|1004          |2         |2026-01-22|450.0 |Ropa       |1                      |
|1005          |2         |2026-04-11|780.0 |Hogar      |2                      |
|1006          |3         |2026-02-03|3200.0|Electrónica|1                      |
|1007          |3         |2026-03-17|1800.0|Ropa       |2                      |
|1008          |3         |2026-06-21|4100.0|Electrónica|3                      |
|1009          |4         |2026-03-04|350.0 |Hogar      |1                      |
|1010          |

El resultado será similar a:

```text
cliente | fecha       | num_transaccion_cliente
--------|-------------|------------------------
1       | 2026-01-05  | 1
1       | 2026-02-18  | 2
1       | 2026-05-09  | 3
```

Esto resulta útil para identificar:

- primera transacción;
- última transacción;
- orden de eventos;
- registros más recientes.

## 36. Identificar la primera transacción de cada cliente

Podemos aprovechar `row_number()`.

In [45]:
df_primera_compra = (
    df_transacciones
    .withColumn(
        "rn",
        F.row_number().over(
            ventana_cliente_fecha
        )
    )
    .filter(
        F.col("rn") == 1
    )
    .drop("rn")
)

df_primera_compra.show(
    truncate=False
)

+--------------+----------+----------+------+-----------+
|id_transaccion|id_cliente|fecha     |monto |categoria  |
+--------------+----------+----------+------+-----------+
|1001          |1         |2026-01-05|1200.0|Electrónica|
|1004          |2         |2026-01-22|450.0 |Ropa       |
|1006          |3         |2026-02-03|3200.0|Electrónica|
|1009          |4         |2026-03-04|350.0 |Hogar      |
|1010          |5         |2026-01-30|650.0 |Ropa       |
|1012          |6         |2026-02-27|2700.0|Electrónica|
|1014          |7         |2026-04-08|1450.0|Electrónica|
|1016          |8         |2026-03-26|520.0 |Ropa       |
|1017          |9         |2026-02-14|1300.0|Hogar      |
|1019          |10        |2026-01-12|2900.0|Electrónica|
|1022          |99        |2026-06-30|5000.0|Electrónica|
+--------------+----------+----------+------+-----------+



Este patrón es muy común para obtener el primer o último registro de cada grupo.

## 37. Ranking por monto

Ahora queremos ordenar las compras de cada cliente de mayor a menor.

In [46]:
ventana_cliente_monto = (
    Window
    .partitionBy("id_cliente")
    .orderBy(
        F.col("monto").desc()
    )
)

Aplicamos:

In [47]:
df_ranking = (
    df_transacciones
    .withColumn(
        "ranking_compra",
        F.rank().over(
            ventana_cliente_monto
        )
    )
)

df_ranking.show(
    truncate=False
)

+--------------+----------+----------+------+-----------+--------------+
|id_transaccion|id_cliente|fecha     |monto |categoria  |ranking_compra|
+--------------+----------+----------+------+-----------+--------------+
|1003          |1         |2026-05-09|2200.0|Electrónica|1             |
|1001          |1         |2026-01-05|1200.0|Electrónica|2             |
|1002          |1         |2026-02-18|850.0 |Hogar      |3             |
|1005          |2         |2026-04-11|780.0 |Hogar      |1             |
|1004          |2         |2026-01-22|450.0 |Ropa       |2             |
|1008          |3         |2026-06-21|4100.0|Electrónica|1             |
|1006          |3         |2026-02-03|3200.0|Electrónica|2             |
|1007          |3         |2026-03-17|1800.0|Ropa       |3             |
|1009          |4         |2026-03-04|350.0 |Hogar      |1             |
|1011          |5         |2026-05-16|900.0 |Ropa       |1             |
|1010          |5         |2026-01-30|650.0 |Ropa  

## 38. `rank()` vs `dense_rank()` vs `row_number()`

Estas tres funciones pueden parecer similares, pero reaccionan de forma distinta cuando existen empates.

Supongamos:

```text
Monto
-----
3000
3000
2000
1000
```

Con `row_number()`:

```text
3000 → 1
3000 → 2
2000 → 3
1000 → 4
```

Con `rank()`:

```text
3000 → 1
3000 → 1
2000 → 3
1000 → 4
```

Con `dense_rank()`:

```text
3000 → 1
3000 → 1
2000 → 2
1000 → 3
```

Podemos compararlas en Spark.

In [48]:
df_rankings = (
    df_transacciones
    .withColumn(
        "row_number",
        F.row_number().over(
            ventana_cliente_monto
        )
    )
    .withColumn(
        "rank",
        F.rank().over(
            ventana_cliente_monto
        )
    )
    .withColumn(
        "dense_rank",
        F.dense_rank().over(
            ventana_cliente_monto
        )
    )
)

df_rankings.show(
    truncate=False
)

+--------------+----------+----------+------+-----------+----------+----+----------+
|id_transaccion|id_cliente|fecha     |monto |categoria  |row_number|rank|dense_rank|
+--------------+----------+----------+------+-----------+----------+----+----------+
|1003          |1         |2026-05-09|2200.0|Electrónica|1         |1   |1         |
|1001          |1         |2026-01-05|1200.0|Electrónica|2         |2   |2         |
|1002          |1         |2026-02-18|850.0 |Hogar      |3         |3   |3         |
|1005          |2         |2026-04-11|780.0 |Hogar      |1         |1   |1         |
|1004          |2         |2026-01-22|450.0 |Ropa       |2         |2   |2         |
|1008          |3         |2026-06-21|4100.0|Electrónica|1         |1   |1         |
|1006          |3         |2026-02-03|3200.0|Electrónica|2         |2   |2         |
|1007          |3         |2026-03-17|1800.0|Ropa       |3         |3   |3         |
|1009          |4         |2026-03-04|350.0 |Hogar      |1       

## 39. Acumulados

Las ventanas también permiten construir métricas acumuladas.

Para ello definimos un rango de filas dentro de la ventana.

In [49]:
ventana_acumulada = (
    Window
    .partitionBy("id_cliente")
    .orderBy("fecha")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

Esto significa:

```text
desde la primera fila del cliente
hasta la fila actual
```

Ahora calculamos el gasto acumulado.

In [50]:
df_acumulado = (
    df_transacciones
    .withColumn(
        "gasto_acumulado",
        F.sum("monto").over(
            ventana_acumulada
        )
    )
)

df_acumulado.show(
    truncate=False
)

+--------------+----------+----------+------+-----------+---------------+
|id_transaccion|id_cliente|fecha     |monto |categoria  |gasto_acumulado|
+--------------+----------+----------+------+-----------+---------------+
|1001          |1         |2026-01-05|1200.0|Electrónica|1200.0         |
|1002          |1         |2026-02-18|850.0 |Hogar      |2050.0         |
|1003          |1         |2026-05-09|2200.0|Electrónica|4250.0         |
|1004          |2         |2026-01-22|450.0 |Ropa       |450.0          |
|1005          |2         |2026-04-11|780.0 |Hogar      |1230.0         |
|1006          |3         |2026-02-03|3200.0|Electrónica|3200.0         |
|1007          |3         |2026-03-17|1800.0|Ropa       |5000.0         |
|1008          |3         |2026-06-21|4100.0|Electrónica|9100.0         |
|1009          |4         |2026-03-04|350.0 |Hogar      |350.0          |
|1010          |5         |2026-01-30|650.0 |Ropa       |650.0          |
|1011          |5         |2026-05-16|

Para un cliente podríamos obtener:

```text
fecha       | monto | gasto_acumulado
------------|-------|----------------
2026-01-05  | 1200  | 1200
2026-02-18  | 850   | 2050
2026-05-09  | 2200  | 4250
```

## 40. Conteo acumulado de transacciones

Podemos hacer lo mismo con el número de compras.

In [52]:
df_acumulado = (
    df_acumulado
    .withColumn(
        "num_compras_acumuladas",
        F.count("*").over(
            ventana_acumulada
        )
    )
)

df_acumulado.show(
    truncate=False
)

+--------------+----------+----------+------+-----------+---------------+----------------------+
|id_transaccion|id_cliente|fecha     |monto |categoria  |gasto_acumulado|num_compras_acumuladas|
+--------------+----------+----------+------+-----------+---------------+----------------------+
|1001          |1         |2026-01-05|1200.0|Electrónica|1200.0         |1                     |
|1002          |1         |2026-02-18|850.0 |Hogar      |2050.0         |2                     |
|1003          |1         |2026-05-09|2200.0|Electrónica|4250.0         |3                     |
|1004          |2         |2026-01-22|450.0 |Ropa       |450.0          |1                     |
|1005          |2         |2026-04-11|780.0 |Hogar      |1230.0         |2                     |
|1006          |3         |2026-02-03|3200.0|Electrónica|3200.0         |1                     |
|1007          |3         |2026-03-17|1800.0|Ropa       |5000.0         |2                     |
|1008          |3         |202

Esto permite construir variables relacionadas con evolución temporal.

## 41. `lag()`

`lag()` permite acceder al valor de una fila anterior dentro de la ventana.

Por ejemplo, queremos conocer el monto de la compra anterior.

In [53]:
df_lag = (
    df_transacciones
    .withColumn(
        "monto_anterior",
        F.lag(
            "monto",
            1
        ).over(
            ventana_cliente_fecha
        )
    )
)

df_lag.show(
    truncate=False
)

+--------------+----------+----------+------+-----------+--------------+
|id_transaccion|id_cliente|fecha     |monto |categoria  |monto_anterior|
+--------------+----------+----------+------+-----------+--------------+
|1001          |1         |2026-01-05|1200.0|Electrónica|NULL          |
|1002          |1         |2026-02-18|850.0 |Hogar      |1200.0        |
|1003          |1         |2026-05-09|2200.0|Electrónica|850.0         |
|1004          |2         |2026-01-22|450.0 |Ropa       |NULL          |
|1005          |2         |2026-04-11|780.0 |Hogar      |450.0         |
|1006          |3         |2026-02-03|3200.0|Electrónica|NULL          |
|1007          |3         |2026-03-17|1800.0|Ropa       |3200.0        |
|1008          |3         |2026-06-21|4100.0|Electrónica|1800.0        |
|1009          |4         |2026-03-04|350.0 |Hogar      |NULL          |
|1010          |5         |2026-01-30|650.0 |Ropa       |NULL          |
|1011          |5         |2026-05-16|900.0 |Ropa  

La primera transacción de cada cliente tendrá:

```text
null
```

porque no existe una compra anterior.

### 41.1 Diferencia respecto a la compra anterior

Podemos calcular:

In [54]:
df_lag = (
    df_lag
    .withColumn(
        "diferencia_monto",
        F.col("monto")
        - F.col("monto_anterior")
    )
)

df_lag.show(
    truncate=False
)

+--------------+----------+----------+------+-----------+--------------+----------------+
|id_transaccion|id_cliente|fecha     |monto |categoria  |monto_anterior|diferencia_monto|
+--------------+----------+----------+------+-----------+--------------+----------------+
|1001          |1         |2026-01-05|1200.0|Electrónica|NULL          |NULL            |
|1002          |1         |2026-02-18|850.0 |Hogar      |1200.0        |-350.0          |
|1003          |1         |2026-05-09|2200.0|Electrónica|850.0         |1350.0          |
|1004          |2         |2026-01-22|450.0 |Ropa       |NULL          |NULL            |
|1005          |2         |2026-04-11|780.0 |Hogar      |450.0         |330.0           |
|1006          |3         |2026-02-03|3200.0|Electrónica|NULL          |NULL            |
|1007          |3         |2026-03-17|1800.0|Ropa       |3200.0        |-1400.0         |
|1008          |3         |2026-06-21|4100.0|Electrónica|1800.0        |2300.0          |
|1009     

Esto es útil para analizar cambios entre eventos consecutivos.


## 42. `lead()`

`lead()` funciona de forma similar, pero permite acceder a la siguiente fila.

In [55]:
df_lead = (
    df_transacciones
    .withColumn(
        "siguiente_monto",
        F.lead(
            "monto",
            1
        ).over(
            ventana_cliente_fecha
        )
    )
)

df_lead.show(
    truncate=False
)

+--------------+----------+----------+------+-----------+---------------+
|id_transaccion|id_cliente|fecha     |monto |categoria  |siguiente_monto|
+--------------+----------+----------+------+-----------+---------------+
|1001          |1         |2026-01-05|1200.0|Electrónica|850.0          |
|1002          |1         |2026-02-18|850.0 |Hogar      |2200.0         |
|1003          |1         |2026-05-09|2200.0|Electrónica|NULL           |
|1004          |2         |2026-01-22|450.0 |Ropa       |780.0          |
|1005          |2         |2026-04-11|780.0 |Hogar      |NULL           |
|1006          |3         |2026-02-03|3200.0|Electrónica|1800.0         |
|1007          |3         |2026-03-17|1800.0|Ropa       |4100.0         |
|1008          |3         |2026-06-21|4100.0|Electrónica|NULL           |
|1009          |4         |2026-03-04|350.0 |Hogar      |NULL           |
|1010          |5         |2026-01-30|650.0 |Ropa       |900.0          |
|1011          |5         |2026-05-16|

Podemos pensar:

```text
lag()
 ↓
fila anterior


lead()
 ↓
fila siguiente
```

## 43. Tiempo entre transacciones

Podemos utilizar `lag()` sobre fechas para calcular cuántos días transcurrieron entre compras consecutivas.

In [56]:
df_intervalos = (
    df_transacciones
    .withColumn(
        "fecha_anterior",
        F.lag(
            "fecha",
            1
        ).over(
            ventana_cliente_fecha
        )
    )
    .withColumn(
        "dias_desde_compra_anterior",
        F.datediff(
            F.col("fecha"),
            F.col("fecha_anterior")
        )
    )
)

df_intervalos.show(
    truncate=False
)

+--------------+----------+----------+------+-----------+--------------+--------------------------+
|id_transaccion|id_cliente|fecha     |monto |categoria  |fecha_anterior|dias_desde_compra_anterior|
+--------------+----------+----------+------+-----------+--------------+--------------------------+
|1001          |1         |2026-01-05|1200.0|Electrónica|NULL          |NULL                      |
|1002          |1         |2026-02-18|850.0 |Hogar      |2026-01-05    |44                        |
|1003          |1         |2026-05-09|2200.0|Electrónica|2026-02-18    |80                        |
|1004          |2         |2026-01-22|450.0 |Ropa       |NULL          |NULL                      |
|1005          |2         |2026-04-11|780.0 |Hogar      |2026-01-22    |79                        |
|1006          |3         |2026-02-03|3200.0|Electrónica|NULL          |NULL                      |
|1007          |3         |2026-03-17|1800.0|Ropa       |2026-02-03    |42                        |


Este tipo de variable puede ser especialmente útil en análisis de comportamiento.

## 44. Última transacción por cliente

Podemos invertir el orden de la ventana.

In [57]:
ventana_ultima = (
    Window
    .partitionBy("id_cliente")
    .orderBy(
        F.col("fecha").desc()
    )
)

Después:

In [58]:
df_ultima_transaccion = (
    df_transacciones
    .withColumn(
        "rn",
        F.row_number().over(
            ventana_ultima
        )
    )
    .filter(
        F.col("rn") == 1
    )
    .drop("rn")
)

df_ultima_transaccion.show(
    truncate=False
)

+--------------+----------+----------+------+-----------+
|id_transaccion|id_cliente|fecha     |monto |categoria  |
+--------------+----------+----------+------+-----------+
|1003          |1         |2026-05-09|2200.0|Electrónica|
|1005          |2         |2026-04-11|780.0 |Hogar      |
|1008          |3         |2026-06-21|4100.0|Electrónica|
|1009          |4         |2026-03-04|350.0 |Hogar      |
|1011          |5         |2026-05-16|900.0 |Ropa       |
|1013          |6         |2026-07-01|1950.0|Hogar      |
|1015          |7         |2026-06-12|2300.0|Electrónica|
|1016          |8         |2026-03-26|520.0 |Ropa       |
|1018          |9         |2026-05-28|1700.0|Electrónica|
|1021          |10        |2026-07-15|3500.0|Electrónica|
|1022          |99        |2026-06-30|5000.0|Electrónica|
+--------------+----------+----------+------+-----------+



Este patrón aparece constantemente en pipelines donde queremos:

```text
registro más reciente
por cliente
```

## 45. Window + Join

Ahora combinaremos varias técnicas del notebook.

Queremos construir una tabla con:

- cada transacción;
- nombre del cliente;
- gasto total del cliente;
- número secuencial de compra;
- gasto acumulado;
- monto anterior;
- días desde la compra anterior.

Primero realizamos el join.

In [59]:
df_transacciones_clientes = (
    df_transacciones
    .join(
        df_clientes,
        on="id_cliente",
        how="left"
    )
)

Definimos las ventanas:

In [60]:
w_cliente = (
    Window
    .partitionBy("id_cliente")
)

In [61]:
w_cliente_fecha = (
    Window
    .partitionBy("id_cliente")
    .orderBy("fecha")
)

In [62]:
w_acumulada = (
    Window
    .partitionBy("id_cliente")
    .orderBy("fecha")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

Construimos las variables:

In [63]:
df_analisis = (
    df_transacciones_clientes

    .withColumn(
        "gasto_total_cliente",
        F.sum("monto").over(
            w_cliente
        )
    )

    .withColumn(
        "numero_compra",
        F.row_number().over(
            w_cliente_fecha
        )
    )

    .withColumn(
        "gasto_acumulado",
        F.sum("monto").over(
            w_acumulada
        )
    )

    .withColumn(
        "monto_anterior",
        F.lag(
            "monto",
            1
        ).over(
            w_cliente_fecha
        )
    )

    .withColumn(
        "fecha_anterior",
        F.lag(
            "fecha",
            1
        ).over(
            w_cliente_fecha
        )
    )

    .withColumn(
        "dias_desde_compra_anterior",
        F.datediff(
            F.col("fecha"),
            F.col("fecha_anterior")
        )
    )

    .select(
        "id_cliente",
        "nombre",
        "estado",
        "id_transaccion",
        "fecha",
        "categoria",
        "monto",
        "numero_compra",
        "gasto_acumulado",
        "gasto_total_cliente",
        "monto_anterior",
        "dias_desde_compra_anterior"
    )

    .orderBy(
        "id_cliente",
        "fecha"
    )
)

df_analisis.show(
    truncate=False
)

+----------+------+----------+--------------+----------+-----------+------+-------------+---------------+-------------------+--------------+--------------------------+
|id_cliente|nombre|estado    |id_transaccion|fecha     |categoria  |monto |numero_compra|gasto_acumulado|gasto_total_cliente|monto_anterior|dias_desde_compra_anterior|
+----------+------+----------+--------------+----------+-----------+------+-------------+---------------+-------------------+--------------+--------------------------+
|1         |Ana   |CDMX      |1001          |2026-01-05|Electrónica|1200.0|1            |1200.0         |4250.0             |NULL          |NULL                      |
|1         |Ana   |CDMX      |1002          |2026-02-18|Hogar      |850.0 |2            |2050.0         |4250.0             |1200.0        |44                        |
|1         |Ana   |CDMX      |1003          |2026-05-09|Electrónica|2200.0|3            |4250.0         |4250.0             |850.0         |80                  

Ahora tenemos un dataset mucho más cercano a una tabla analítica real.

## 46. Window Functions y shuffle

Las Window Functions suelen requerir que Spark:

1. agrupe registros por las columnas definidas en `partitionBy()`;
2. ordene los registros cuando utilizamos `orderBy()`.

Esto puede implicar operaciones costosas como:

```text
Exchange
Sort
Window
```

Podemos inspeccionarlo:

In [64]:
df_analisis.explain(
    "formatted"
)

== Physical Plan ==
AdaptiveSparkPlan (17)
+- Sort (16)
   +- Exchange (15)
      +- Project (14)
         +- Window (13)
            +- Sort (12)
               +- Window (11)
                  +- Project (10)
                     +- SortMergeJoin LeftOuter (9)
                        :- Sort (4)
                        :  +- Exchange (3)
                        :     +- Project (2)
                        :        +- Scan ExistingRDD (1)
                        +- Sort (8)
                           +- Exchange (7)
                              +- Project (6)
                                 +- Scan ExistingRDD (5)


(1) Scan ExistingRDD
Output [5]: [id_transaccion#17, id_cliente#18, fecha#19, monto#20, categoria#21]
Arguments: [id_transaccion#17, id_cliente#18, fecha#19, monto#20, categoria#21], MapPartitionsRDD[11] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Project
Output [5]: [id_transaccion#17, id_cliente#18, cast(gettim

Busca elementos relacionados con:

```text
Exchange
Sort
Window
```

Esto conecta nuevamente las operaciones de alto nivel que escribimos en PySpark con el modelo físico estudiado en `02_01`.

## 47. `groupBy()` vs Window Functions

Podemos resumir la diferencia:

| Característica | `groupBy()` | Window |
|---|---|---|
| Reduce filas | Sí | No |
| Mantiene detalle | No | Sí |
| Calcula métricas por grupo | Sí | Sí |
| Permite rankings | No directamente | Sí |
| Permite acumulados | No directamente | Sí |
| Permite `lag()` / `lead()` | No | Sí |
| Requiere particionar por claves | Frecuentemente | Frecuentemente |
| Puede implicar shuffle | Sí | Sí |

Una forma sencilla de decidir es:

```text
¿Necesito una fila por grupo?
        ↓
     groupBy()


¿Necesito conservar las filas
pero agregar información del grupo?
        ↓
      Window
```

## 48. Ejercicio integrador

Construye un DataFrame llamado:

```python
df_historial_cliente
```

que contenga:

```text
id_cliente
nombre
id_transaccion
fecha
monto
numero_compra
monto_anterior
diferencia_monto
gasto_acumulado
gasto_total_cliente
```

Condiciones:

1. relacionar clientes y transacciones;
2. ordenar las compras cronológicamente dentro de cada cliente;
3. numerar cada compra;
4. obtener el monto anterior;
5. calcular la diferencia contra la compra anterior;
6. calcular el gasto acumulado;
7. calcular el gasto total de cada cliente.

### 48.1 Solución

In [65]:
w_cliente = (
    Window
    .partitionBy("id_cliente")
)

w_fecha = (
    Window
    .partitionBy("id_cliente")
    .orderBy("fecha")
)

w_acum = (
    Window
    .partitionBy("id_cliente")
    .orderBy("fecha")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

df_historial_cliente = (
    df_transacciones

    .join(
        df_clientes,
        on="id_cliente",
        how="left"
    )

    .withColumn(
        "numero_compra",
        F.row_number().over(
            w_fecha
        )
    )

    .withColumn(
        "monto_anterior",
        F.lag(
            "monto"
        ).over(
            w_fecha
        )
    )

    .withColumn(
        "diferencia_monto",
        F.col("monto")
        - F.col("monto_anterior")
    )

    .withColumn(
        "gasto_acumulado",
        F.sum(
            "monto"
        ).over(
            w_acum
        )
    )

    .withColumn(
        "gasto_total_cliente",
        F.sum(
            "monto"
        ).over(
            w_cliente
        )
    )

    .select(
        "id_cliente",
        "nombre",
        "id_transaccion",
        "fecha",
        "monto",
        "numero_compra",
        "monto_anterior",
        "diferencia_monto",
        "gasto_acumulado",
        "gasto_total_cliente"
    )

    .orderBy(
        "id_cliente",
        "fecha"
    )
)

df_historial_cliente.show(
    truncate=False
)

+----------+------+--------------+----------+------+-------------+--------------+----------------+---------------+-------------------+
|id_cliente|nombre|id_transaccion|fecha     |monto |numero_compra|monto_anterior|diferencia_monto|gasto_acumulado|gasto_total_cliente|
+----------+------+--------------+----------+------+-------------+--------------+----------------+---------------+-------------------+
|1         |Ana   |1001          |2026-01-05|1200.0|1            |NULL          |NULL            |1200.0         |4250.0             |
|1         |Ana   |1002          |2026-02-18|850.0 |2            |1200.0        |-350.0          |2050.0         |4250.0             |
|1         |Ana   |1003          |2026-05-09|2200.0|3            |850.0         |1350.0          |4250.0         |4250.0             |
|2         |Luis  |1004          |2026-01-22|450.0 |1            |NULL          |NULL            |450.0          |1230.0             |
|2         |Luis  |1005          |2026-04-11|780.0 |2  

## 49. Conclusiones

En este notebook trabajamos con tres grupos de operaciones fundamentales en Spark:

```text
Agregaciones
     +
Joins
     +
Window Functions
```

Con agregaciones aprendimos a resumir información mediante:

```text
groupBy()
agg()
count()
sum()
avg()
min()
max()
countDistinct()
```

Con joins relacionamos diferentes fuentes utilizando:

```text
inner
left
right
full
left_semi
left_anti
```

También vimos la importancia de utilizar aliases y seleccionar explícitamente columnas para evitar ambigüedades.

Finalmente, con Window Functions aprendimos a calcular información por grupo sin perder el detalle individual de las filas.

Utilizamos:

```text
row_number()
rank()
dense_rank()
sum().over()
count().over()
lag()
lead()
```

y construimos métricas como:

- ranking de compras;
- gasto acumulado;
- gasto total por cliente;
- compra anterior;
- días entre compras;
- primera y última transacción.

Estas operaciones forman una parte muy importante del trabajo cotidiano con Spark porque permiten transformar datos transaccionales en datasets analíticos y features para modelos.